# LandTrendr Analysis for Austin Urban Growth
## UT Austin Geoscience Hackathon: AlphaEarthHack Team

This notebook presents a comprehensive analysis of land use change in Austin, Texas using Google Earth Engine's LandTrendr algorithm. Our team has developed multiple assets and analysis tools to understand urban growth patterns from 2016-2024.

## Project Overview

The AlphaEarthHack team has created a suite of tools and analyses to study environmental changes in the Austin metropolitan area using satellite imagery and machine learning techniques. This work was completed as part of the UT Austin Geoscience Hackathon.

### Key Objectives:
- **Urban Growth Analysis**: Track expansion of urban areas in Austin using LandTrendr
- **NDVI Time Series**: Analyze vegetation changes over time
- **Interactive Visualization**: Create dynamic maps for exploring land use changes
- **Data Export**: Generate processed datasets for further analysis

---

## 🎯 **Getting Started**

### Prerequisites
```bash
# Install required packages
pip install earthengine-api geemap

# Authenticate Earth Engine (first time only)
earthengine authenticate
```

### Quick Start Guide

1. **Explore the Data**: Start with `AlphaEarth_EDA.ipynb` for initial data exploration
2. **Run Analysis**: Execute this notebook (`LandTrendr_AlphaEarth.ipynb`) for complete analysis  
3. **Focus on Changes**: Use `Austin_LandTrendr_Changes.ipynb` for detailed change analysis
4. **Optimize Parameters**: Reference `LandTrendr_EDA.ipynb` for algorithm tuning
5. **Batch Processing**: Run `Austin_LandTrendr.py` for automated data processing

### Data Access
- **Earth Engine Assets**: Pre-processed LandTrendr results stored in Google Earth Engine
- **Local Data**: `austin_lt_dur_2016_2024.tif` contains duration analysis results
- **Interactive Maps**: Generated dynamically in this notebook using geemap

---

## 🏆 **AlphaEarthHack Team Contributions**

This comprehensive analysis demonstrates:
- **Technical Innovation**: Integration of multiple Earth Engine algorithms
- **Methodological Rigor**: Systematic approach to change detection validation  
- **Reproducible Research**: Well-documented code and clear asset organization
- **Practical Applications**: Focus on real-world urban planning insights

*Ready to explore Austin's urban growth story? Let's dive into the analysis!*

In [1]:
import ee
import geemap

# Initialize Earth Engine
ee.Initialize()

# All LandTrendr assets
assets = {
    'Angelina_Forest': 'users/xihanyao/LT_Angelina_Forest_TX_31_95_NDVI_2016_2024',
    'Austin_Urban': 'users/xihanyao/LT_Austin_UrbanGrowth_30_98_NDVI_2016_2024', 
    'Bend_Urban': 'users/xihanyao/LT_Bend_UrbanExpansion_44_121_NDVI_2016_2024',
    'Bootleg_Fire': 'users/xihanyao/LT_Bootleg_Fire_2021_43_121_NDVI_2016_2024',
    'CampFire_CA': 'users/xihanyao/LT_CampFire_CA_2018_40_122_NDVI_2016_2024',
    'CoosBay_Forest': 'users/xihanyao/LT_CoosBay_IndustrialForestry_43_124_NDVI_2016_2024',
    'Dallas_TX': 'users/xihanyao/LT_Dallas_TX_33_97_NDVI_2016_2024',
    'DixieFire_CA': 'users/xihanyao/LT_DixieFire_CA_2021_40_121_NDVI_2016_2024',
    'Houston_TX': 'users/xihanyao/LT_Houston_TX_30_95_NDVI_2016_2024',
    'MosquitoFire_CA': 'users/xihanyao/LT_MosquitoFire_CA_2022_39_121_NDVI_2016_2024',
    'MtHood_Forest': 'users/xihanyao/LT_MtHood_WUI_Forestry_45_122_NDVI_2016_2024',
    'Portland_Urban': 'users/xihanyao/LT_Portland_Metro_UrbanGrowth_45_123_NDVI_2016_2024',
    'Sacramento_Urban': 'users/xihanyao/LT_Sacramento_UrbanEdge_39_121_NDVI_2016_2024',
    'Santiam_Fire': 'users/xihanyao/LT_Santiam_Fire_2020_45_122_NDVI_2016_2024',
    'ShastaTrinity_Forest': 'users/xihanyao/LT_ShastaTrinity_Timberlands_41_122_NDVI_2016_2024'
}

# Load all images
images = {name: ee.Image(path) for name, path in assets.items()}

print(f"✅ Loaded {len(assets)} LandTrendr assets")
for name in assets.keys():
    print(f"  • {name}")

# Create map centered over western US
Map = geemap.Map(center=[30.2672, -97.7431], zoom=9)

✅ Loaded 15 LandTrendr assets
  • Angelina_Forest
  • Austin_Urban
  • Bend_Urban
  • Bootleg_Fire
  • CampFire_CA
  • CoosBay_Forest
  • Dallas_TX
  • DixieFire_CA
  • Houston_TX
  • MosquitoFire_CA
  • MtHood_Forest
  • Portland_Urban
  • Sacramento_Urban
  • Santiam_Fire
  • ShastaTrinity_Forest


In [2]:
# # Color palettes for different study types
# palettes = {
#     'Urban': ['#0C2C84', '#41B6C4', '#FFFFCC', '#FEB24C', '#B10026'],
#     'Fire': ['#8B0000', '#FF4500', '#FFD700', '#ADFF2F', '#006400'], 
#     'Forest': ['#654321', '#8B4513', '#DEB887', '#90EE90', '#228B22']
# }

# # Categorize assets by type (kept for markers below)
# categories = {
#     'Urban': ['Austin_Urban', 'Bend_Urban', 'Dallas_TX', 'Houston_TX', 'Portland_Urban', 'Sacramento_Urban'],
#     'Fire': ['Bootleg_Fire', 'CampFire_CA', 'DixieFire_CA', 'MosquitoFire_CA', 'Santiam_Fire'],
#     'Forest': ['Angelina_Forest', 'CoosBay_Forest', 'MtHood_Forest', 'ShastaTrinity_Forest']
# }

# --- Simple visualization: one color scheme for all assets ---
single_palette = ['#440154', '#31688e', '#35b779', '#fde725']  # Viridis-like

def vis_for_band(band):
    b = band.lower()
    if 'yod' in b:
        return {'min': 2016, 'max': 2024, 'palette': single_palette}
    if 'mag' in b:
        return {'min': -1000, 'max': 1000, 'palette': single_palette}
    if 'dur' in b:
        return {'min': 1, 'max': 8, 'palette': single_palette}
    return {'min': 0, 'max': 100, 'palette': single_palette}

# Add one layer per asset using preferred band order (yod -> mag -> dur)
def pick_band_by_key(img, key):
    names = img.bandNames().getInfo() or []
    for b in names:
        if key in b.lower():
            return b
    return None

# Add MAG for each asset (fallback: print available bands)
for name, img in images.items():
    mag_band = pick_band_by_key(img, 'mag')
    if mag_band:
        Map.addLayer(img.select(mag_band), vis_for_band(mag_band), f"{name} (mag)", True)
    else:
        print(f"[{name}] 'mag' band not found. Bands: {img.bandNames().getInfo()}")

print(f"✅ Added {len(images)} asset layers with a single color scheme")
print("📍 No category styling; showing magnitude by default")

✅ Added 15 asset layers with a single color scheme
📍 No category styling; showing magnitude by default


In [3]:
# Study area coordinates
locations = {
    'Austin_Urban': [30.27, -97.74], 'Bend_Urban': [44.06, -121.31], 'Dallas_TX': [32.78, -96.80],
    'Houston_TX': [29.76, -95.37], 'Portland_Urban': [45.52, -122.68], 'Sacramento_Urban': [38.58, -121.49],
    'Bootleg_Fire': [42.65, -120.85], 'CampFire_CA': [39.81, -121.44], 'DixieFire_CA': [40.17, -121.11],
    'MosquitoFire_CA': [38.97, -120.72], 'Santiam_Fire': [44.72, -122.03], 
    'Angelina_Forest': [31.30, -94.60], 'CoosBay_Forest': [43.37, -124.22],
    'MtHood_Forest': [45.37, -121.71], 'ShastaTrinity_Forest': [40.74, -122.44]
}

# Add study area markers (single color)
marker_color = 'yellow'
for asset_name, coords in locations.items():
    point = ee.Geometry.Point([coords[1], coords[0]]).buffer(5000)
    Map.addLayer(point, {'color': marker_color}, f"{asset_name} Area", True)

# Map.add_basemap('SATELLITE')
print("🗺️ Interactive map ready with all LandTrendr datasets")
print("🎨 Markers: single color; categories removed")
Map

🗺️ Interactive map ready with all LandTrendr datasets
🎨 Markers: single color; categories removed


Map(center=[30.2672, -97.7431], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

In [4]:
# Dataset summary (no categories)
print("📊 COMPREHENSIVE LANDTRENDR ANALYSIS")
print(f"Total Assets: {len(assets)}")
print(f"Locations with coordinates: {len(locations)}")

print("\n🌎 Geographic Coverage:")
print("• Texas: Austin, Dallas, Houston, Angelina Forest")
print("• California: Camp Fire, Dixie Fire, Mosquito Fire, Sacramento, Shasta-Trinity")
print("• Oregon: Bootleg Fire, Santiam Fire, Portland, Coos Bay, Mt Hood, Bend")

# Quick access dictionary (image + location only)
all_datasets = {
    name: {'image': images.get(name), 'location': locations.get(name)}
    for name in assets.keys() if images.get(name)
}

print(f"\n💾 {len(all_datasets)} datasets available in 'all_datasets' dictionary")

📊 COMPREHENSIVE LANDTRENDR ANALYSIS
Total Assets: 15
Locations with coordinates: 15

🌎 Geographic Coverage:
• Texas: Austin, Dallas, Houston, Angelina Forest
• California: Camp Fire, Dixie Fire, Mosquito Fire, Sacramento, Shasta-Trinity
• Oregon: Bootleg Fire, Santiam Fire, Portland, Coos Bay, Mt Hood, Bend

💾 15 datasets available in 'all_datasets' dictionary


---

## 🔍 **Comparative Change Detection Analysis**

Now we'll apply comprehensive change detection to highlight areas of significant changes across all 15 datasets. This analysis includes:

- **Smoothing**: Gaussian kernel to reduce noise
- **Magnitude Thresholding**: Filter significant changes (>170 magnitude)  
- **Morphological Operations**: Cluster nearby change pixels
- **Training Labels**: Create clean change masks for all YOD/MAG/DUR bands

This workflow identifies the most significant land use changes for comparative analysis across urban growth, fire events, and forest changes.

In [5]:
# =============================================================================
# CHANGE DETECTION WORKFLOW FOR ALL ASSETS
# =============================================================================

def detect_significant_changes(image, asset_name, magnitude_threshold=170):
    """
    Apply smoothing, thresholding, and morphological operations to detect significant changes.
    Based on Austin_LandTrendr_Changes.ipynb workflow.
    """
    
    # Get bands
    bands = image.bandNames().getInfo()
    mag_band = next((b for b in bands if 'mag' in b.lower()), bands[0] if bands else None)
    
    if not mag_band:
        print(f"[{asset_name}] No bands found, skipping...")
        return None, None
    
    # 1. Apply smoothing kernel to magnitude band
    kernel = ee.Kernel.gaussian(radius=2, sigma=1, units='pixels')
    magnitude_smoothed = image.select(mag_band).convolve(kernel)
    
    # 2. Filter by magnitude threshold 
    significant_change_mask = magnitude_smoothed.abs().gt(magnitude_threshold)
    
    # 3. Morphological operations for clustering
    kernel_close = ee.Kernel.circle(radius=2)
    morphological_closed = significant_change_mask.focalMax(kernel=kernel_close).focalMin(kernel=kernel_close)
    
    kernel_open = ee.Kernel.circle(radius=1)
    training_mask = morphological_closed.focalMin(kernel=kernel_open).focalMax(kernel=kernel_open)
    
    # 4. Create training labels with all bands preserved
    training_labels = image.updateMask(training_mask)
    
    return training_mask, training_labels

print("🔧 Processing all 15 assets for change detection...")

# Process each asset
change_results = {}
for name, img in images.items():
    print(f"Processing {name}...")
    mask, labels = detect_significant_changes(img, name)
    if mask is not None:
        change_results[name] = {
            'change_mask': mask,
            'training_labels': labels,
            'original': img
        }

print(f"✅ Processed {len(change_results)} assets successfully")
print("📊 Results stored in 'change_results' dictionary")

🔧 Processing all 15 assets for change detection...
Processing Angelina_Forest...
Processing Austin_Urban...
Processing Bend_Urban...
Processing Bootleg_Fire...
Processing CampFire_CA...
Processing CoosBay_Forest...
Processing Dallas_TX...
Processing DixieFire_CA...
Processing Houston_TX...
Processing MosquitoFire_CA...
Processing MtHood_Forest...
Processing Portland_Urban...
Processing Sacramento_Urban...
Processing Santiam_Fire...
Processing ShastaTrinity_Forest...
✅ Processed 15 assets successfully
📊 Results stored in 'change_results' dictionary


In [6]:
# =============================================================================
# VISUALIZE SMOOTHED MAGNITUDE BANDS FOR ALL ASSETS
# =============================================================================

# Create a new map for smoothed magnitude visualization
Map_Smoothed = geemap.Map(center=[30.2672, -97.7431], zoom=8)

print("🎨 Creating smoothed magnitude visualizations for all assets...")

# Generate smoothed magnitude layers for each asset
smoothed_results = {}

for name, img in images.items():
    if name in locations:
        print(f"Processing smoothed magnitude for {name}...")
        
        # Get magnitude band
        bands = img.bandNames().getInfo()
        mag_band = next((b for b in bands if 'mag' in b.lower()), bands[0] if bands else None)
        
        if mag_band:
            # Apply Gaussian smoothing kernel (same as change detection)
            kernel = ee.Kernel.gaussian(radius=2, sigma=1, units='pixels')
            magnitude_smoothed = img.select(mag_band).convolve(kernel)
            
            # Store result
            smoothed_results[name] = magnitude_smoothed
            
            # Visualization parameters for smoothed magnitude
            smoothed_vis = {
                'min': -1000,
                'max': 1000,
                'palette': ['#8B0000', '#FF4500', '#FFD700', '#FFFFFF', '#00FF00', '#006400']  # Red-White-Green
            }
            
            # Add to map (start with layers off for performance)
            Map_Smoothed.addLayer(
                magnitude_smoothed, 
                smoothed_vis, 
                f"{name} - Smoothed Magnitude", 
                False
            )

# # Add basemap
# Map_Smoothed.add_basemap('SATELLITE')

# Add location markers for easy navigation
marker_color = 'cyan'
for asset_name, coords in locations.items():
    if asset_name in smoothed_results:
        point = ee.Geometry.Point([coords[1], coords[0]]).buffer(3000)
        Map_Smoothed.addLayer(point, {'color': marker_color}, f"{asset_name} Location", True)

print(f"✅ Created smoothed magnitude layers for {len(smoothed_results)} assets")
print("🎯 Smoothing applied: Gaussian kernel (radius=2, sigma=1)")
print("🎨 Color scheme: Red (negative) → White (neutral) → Green (positive)")
print("📍 Cyan markers show study area locations")
print("💡 Use layer control to toggle individual smoothed magnitude layers")

# Store results for access
globals()['smoothed_results'] = smoothed_results

Map_Smoothed

🎨 Creating smoothed magnitude visualizations for all assets...
Processing smoothed magnitude for Angelina_Forest...
Processing smoothed magnitude for Austin_Urban...
Processing smoothed magnitude for Bend_Urban...
Processing smoothed magnitude for Bootleg_Fire...
Processing smoothed magnitude for CampFire_CA...
Processing smoothed magnitude for CoosBay_Forest...
Processing smoothed magnitude for Dallas_TX...
Processing smoothed magnitude for DixieFire_CA...
Processing smoothed magnitude for Houston_TX...
Processing smoothed magnitude for MosquitoFire_CA...
Processing smoothed magnitude for MtHood_Forest...
Processing smoothed magnitude for Portland_Urban...
Processing smoothed magnitude for Sacramento_Urban...
Processing smoothed magnitude for Santiam_Fire...
Processing smoothed magnitude for ShastaTrinity_Forest...
✅ Created smoothed magnitude layers for 15 assets
🎯 Smoothing applied: Gaussian kernel (radius=2, sigma=1)
🎨 Color scheme: Red (negative) → White (neutral) → Green (positiv

Map(center=[30.2672, -97.7431], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

In [7]:
# =============================================================================
# STATS FOR ALL 15 ASSETS (YOD, MAG, DUR) USING 170 MAG FILTER TRAINING AREAS
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Generating statistics for all assets using training masks (|mag| > 170)...")

# Ensure change_results exists (uses detect_significant_changes with magnitude_threshold=170)
if 'change_results' not in globals() or not change_results:
    print("change_results not found. Building with magnitude_threshold=170...")
    def detect_significant_changes(image, asset_name, magnitude_threshold=170):
        bands = image.bandNames().getInfo()
        mag_band = next((b for b in bands if 'mag' in b.lower()), bands[0] if bands else None)
        if not mag_band:
            print(f"[{asset_name}] No bands found, skipping...")
            return None, None
        kernel = ee.Kernel.gaussian(radius=2, sigma=1, units='pixels')
        magnitude_smoothed = image.select(mag_band).convolve(kernel)
        significant_change_mask = magnitude_smoothed.abs().gt(magnitude_threshold)
        kernel_close = ee.Kernel.circle(radius=2)
        morphological_closed = significant_change_mask.focalMax(kernel=kernel_close).focalMin(kernel=kernel_close)
        kernel_open = ee.Kernel.circle(radius=1)
        training_mask = morphological_closed.focalMin(kernel=kernel_open).focalMax(kernel=kernel_open)
        training_labels = image.updateMask(training_mask)
        return training_mask, training_labels

    change_results = {}
    for name, img in images.items():
        mask, labels = detect_significant_changes(img, name, magnitude_threshold=170)
        if mask is not None:
            change_results[name] = {'change_mask': mask, 'training_labels': labels, 'original': img}
    print(f"Built change_results for {len(change_results)} assets.")

# Helper to find canonical band names
def find_band(names, key):
    return next((b for b in names if key in b.lower()), None)

# Config
sample_scale = 90
sample_pixels = 2000
plot_per_asset = False  # set True to draw per-asset hist/box/heatmap plots

# Collectors
all_band_stats = []
asset_corr_rows = []
asset_summary = []

pixel_area = ee.Image.pixelArea()

for asset, res in change_results.items():
    try:
        img = res['original']
        labels = res['training_labels']
        mask = res['change_mask']

        band_list = labels.bandNames().getInfo() or []
        yod_b = find_band(band_list, 'yod')
        mag_b = find_band(band_list, 'mag')
        dur_b = find_band(band_list, 'dur')
        bands_to_use = [b for b in [yod_b, mag_b, dur_b] if b]

        if not bands_to_use:
            print(f"[{asset}] No YOD/MAG/DUR bands found, skipping.")
            continue

        # Sample masked (training) data
        sample_fc = labels.select(bands_to_use).sample(
            region=img.geometry(),
            scale=sample_scale,
            numPixels=sample_pixels,
            seed=42
        ).getInfo()

        if not sample_fc or 'features' not in sample_fc or not sample_fc['features']:
            print(f"[{asset}] No samples returned.")
            continue

        # Build DataFrame
        rows = []
        for f in sample_fc['features']:
            props = f.get('properties', {})
            row = {b: props.get(b, None) for b in bands_to_use}
            # drop rows with all None
            if any(v is not None for v in row.values()):
                rows.append(row)
        if not rows:
            print(f"[{asset}] All sampled rows empty after filtering.")
            continue

        df = pd.DataFrame(rows).dropna()
        if df.empty:
            print(f"[{asset}] No valid samples after dropna.")
            continue

        # Per-band stats
        for b in bands_to_use:
            vals = df[b].values
            all_band_stats.append({
                'asset': asset,
                'band': b,
                'count': len(vals),
                'mean': float(np.mean(vals)),
                'std': float(np.std(vals)),
                'min': float(np.min(vals)),
                'p25': float(np.percentile(vals, 25)),
                'median': float(np.median(vals)),
                'p75': float(np.percentile(vals, 75)),
                'max': float(np.max(vals))
            })

        # Correlations across available bands (if >=2)
        if df.shape[1] >= 2:
            corr = df.corr()
            # Flatten upper triangle
            cols = list(corr.columns)
            for i in range(len(cols)):
                for j in range(i + 1, len(cols)):
                    asset_corr_rows.append({
                        'asset': asset,
                        'pair': f'{cols[i]}~{cols[j]}',
                        'corr': float(corr.iloc[i, j])
                    })

        # Area (km^2) and training pixel count
        area_info = pixel_area.updateMask(mask).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=img.geometry(),
            scale=30,
            maxPixels=1e9
        ).getInfo() or {}
        area_km2 = None
        if area_info:
            area_km2 = float(list(area_info.values())[0]) / 1e6

        pix_info = mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=img.geometry(),
            scale=30,
            maxPixels=1e9
        ).getInfo() or {}
        training_px = None
        if pix_info:
            training_px = float(list(pix_info.values())[0])

        asset_summary.append({
            'asset': asset,
            'training_pixels': training_px,
            'training_area_km2': area_km2,
            'bands_used': ','.join(bands_to_use)
        })

        # Optional per-asset plotting
        if plot_per_asset:
            fig, axes = plt.subplots(1, len(bands_to_use), figsize=(5*len(bands_to_use), 4))
            if len(bands_to_use) == 1:
                axes = [axes]
            for ax, b in zip(axes, bands_to_use):
                ax.hist(df[b].values, bins=30, alpha=0.8, edgecolor='black')
                ax.set_title(f'{asset} - {b}')
                ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"[{asset}] Error computing stats: {e}")

# Build DataFrames
stats_df = pd.DataFrame(all_band_stats)
corr_df = pd.DataFrame(asset_corr_rows)
summary_df = pd.DataFrame(asset_summary)

print("\n=== STATS SUMMARY (first 20 rows) ===")
print(stats_df.head(20).to_string(index=False))

if not corr_df.empty:
    print("\n=== CORRELATIONS (first 20 rows) ===")
    print(corr_df.head(20).to_string(index=False))
else:
    print("\nNo correlation pairs computed (not enough bands).")

print("\n=== ASSET SUMMARY ===")
print(summary_df.to_string(index=False))

# Save CSVs to workspace
stats_df.to_csv('lt_training_stats_all_assets.csv', index=False)
corr_df.to_csv('lt_training_corr_all_assets.csv', index=False)
summary_df.to_csv('lt_training_asset_summary.csv', index=False)
print("\nSaved:")
print(" - lt_training_stats_all_assets.csv")
print(" - lt_training_corr_all_assets.csv")
print(" - lt_training_asset_summary.csv")

Generating statistics for all assets using training masks (|mag| > 170)...

=== STATS SUMMARY (first 20 rows) ===
          asset band  count        mean        std    min     p25  median     p75    max
Angelina_Forest  yod    190 2018.668421   2.044390 2017.0 2017.00  2018.0 2020.00 2024.0
Angelina_Forest  mag    190  285.284211 138.506672   35.0  174.25   277.5  376.75  744.0
Angelina_Forest  dur    190    4.547368   2.618832    1.0    2.00     5.0    7.75    8.0
   Austin_Urban  yod    149 2018.107383   1.791632 2017.0 2017.00  2017.0 2018.00 2024.0
   Austin_Urban  mag    149  274.590604 146.691157   19.0  165.00   259.0  374.00  728.0
   Austin_Urban  dur    149    4.684564   2.435992    1.0    2.00     5.0    7.00    8.0
     Bend_Urban  yod     75 2018.706667   2.331085 2017.0 2017.00  2018.0 2019.50 2024.0
     Bend_Urban  mag     75  222.786667  67.265304   43.0  190.50   222.0  260.00  398.0
     Bend_Urban  dur     75    3.453333   2.510210    1.0    2.00     2.0    5.00    

In [8]:
# =============================================================================
# SHOW ALL FINAL LABELS (SMOOTHED MAG, MODE YOD/DUR) FOR ALL 15 ASSETS
# =============================================================================
import ee, geemap

def find_band(img, key):
    names = (img.bandNames().getInfo() or [])
    return next((b for b in names if key in b.lower()), None)

def build_categorical_labels(img, magnitude_threshold=170):
    mag_b = find_band(img, 'mag')
    yod_b = find_band(img, 'yod')
    dur_b = find_band(img, 'dur')
    if not mag_b:
        return None

    # Enhanced magnitude smoothing (bias toward large |mag|)
    mag_kernel_large = ee.Kernel.gaussian(radius=4, sigma=2, units='pixels')
    mag_smoothed = img.select(mag_b).convolve(mag_kernel_large)
    mag_abs = img.select(mag_b).abs()
    weight = mag_abs.divide(mag_abs.add(100))
    mag_enhanced = mag_smoothed.multiply(weight).add(
        img.select(mag_b).multiply(ee.Image.constant(1).subtract(weight))
    )

    # Threshold + small-cluster removal
    sig = mag_enhanced.abs().gt(magnitude_threshold)
    small_kernel = ee.Kernel.circle(radius=1.2)
    mask = sig.focalMin(kernel=small_kernel).focalMax(kernel=small_kernel)

    # Categorical smoothing (mode) for YOD and DUR
    yod_cat = img.select(yod_b).focalMode(kernel=ee.Kernel.circle(radius=3)) if yod_b else None
    dur_cat = img.select(dur_b).focalMode(kernel=ee.Kernel.circle(radius=3)) if dur_b else None

    # Apply mask and assemble output bands
    out = []
    if yod_cat:
        out.append(yod_cat.updateMask(mask).rename('yod_categorical'))
    out.append(mag_enhanced.updateMask(mask).rename('mag_enhanced'))
    if dur_cat:
        out.append(dur_cat.updateMask(mask).rename('dur_categorical'))

    return mask, ee.Image.cat(out)

labels_by_asset = {}
for name, img in images.items():
    try:
        result = build_categorical_labels(img, magnitude_threshold=170)
        if result:
            labels_by_asset[name] = {'mask': result[0], 'labels': result[1], 'image': img}
            print(f"✓ Built labels for {name}")
        else:
            print(f"✗ Skipped {name} (no MAG band)")
    except Exception as e:
        print(f"✗ {name}: {e}")

print(f"\nBuilt labels for {len(labels_by_asset)} assets")

# Visualization
Map_AllLabels = geemap.Map(center=[37.5, -98], zoom=4)

vis_mask = {'min': 0, 'max': 1, 'palette': ['white', 'red']}
vis_yod = {'bands': ['yod_categorical'], 'min': 2016, 'max': 2024,
           'palette': ['navy','blue','cyan','lime','yellow','orange','red','darkred']}
vis_mag = {'bands': ['mag_enhanced'], 'min': -800, 'max': 800,
           'palette': ['darkred','red','orange','white','lightgreen','green','darkgreen']}
vis_dur = {'bands': ['dur_categorical'], 'min': 1, 'max': 8,
           'palette': ['yellow','orange','darkorange','red','darkred','purple','navy']}

for name, rec in labels_by_asset.items():
    Map_AllLabels.addLayer(rec['mask'], vis_mask, f"{name} • Mask", False)
    bands = rec['labels'].bandNames().getInfo()
    if 'yod_categorical' in bands:
        Map_AllLabels.addLayer(rec['labels'].select('yod_categorical'), vis_yod, f"{name} • YOD (mode)", False)
    Map_AllLabels.addLayer(rec['labels'].select('mag_enhanced'), vis_mag, f"{name} • MAG (smoothed)", name=='Austin_Urban')
    if 'dur_categorical' in bands:
        Map_AllLabels.addLayer(rec['labels'].select('dur_categorical'), vis_dur, f"{name} • DUR (mode)", False)

print("Map with all labels is ready. Toggle layers per asset to inspect.")
Map_AllLabels

✓ Built labels for Angelina_Forest
✓ Built labels for Austin_Urban
✓ Built labels for Bend_Urban
✓ Built labels for Bootleg_Fire
✓ Built labels for CampFire_CA
✓ Built labels for CoosBay_Forest
✓ Built labels for Dallas_TX
✓ Built labels for DixieFire_CA
✓ Built labels for Houston_TX
✓ Built labels for MosquitoFire_CA
✓ Built labels for MtHood_Forest
✓ Built labels for Portland_Urban
✓ Built labels for Sacramento_Urban
✓ Built labels for Santiam_Fire
✓ Built labels for ShastaTrinity_Forest

Built labels for 15 assets
Map with all labels is ready. Toggle layers per asset to inspect.


Map(center=[37.5, -98], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

In [9]:
# Just display final labels (smoothed MAG, mode YOD/DUR) for all assets
import ee, geemap

def _find_band(img, key):
    names = (img.bandNames().getInfo() or [])
    return next((b for b in names if key in b.lower()), None)

def _build_categorical_labels(img, magnitude_threshold=170):
    mag_b = _find_band(img, 'mag')
    yod_b = _find_band(img, 'yod')
    dur_b = _find_band(img, 'dur')
    if not mag_b:
        return None

    # Enhanced MAG smoothing with bias to large |mag|
    mag_kernel_large = ee.Kernel.gaussian(radius=4, sigma=2, units='pixels')
    mag_smoothed = img.select(mag_b).convolve(mag_kernel_large)
    mag_abs = img.select(mag_b).abs()
    weight = mag_abs.divide(mag_abs.add(100))
    mag_enhanced = mag_smoothed.multiply(weight).add(
        img.select(mag_b).multiply(ee.Image.constant(1).subtract(weight))
    )

    # Threshold and small-cluster removal
    sig = mag_enhanced.abs().gt(magnitude_threshold)
    small_kernel = ee.Kernel.circle(radius=1.2)
    mask = sig.focalMin(kernel=small_kernel).focalMax(kernel=small_kernel)

    # Categorical smoothing (mode) for YOD and DUR
    yod_cat = img.select(yod_b).focalMode(kernel=ee.Kernel.circle(radius=3)) if yod_b else None
    dur_cat = img.select(dur_b).focalMode(kernel=ee.Kernel.circle(radius=3)) if dur_b else None

    # Assemble masked outputs
    out = []
    if yod_cat:
        out.append(yod_cat.updateMask(mask).rename('yod_categorical'))
    out.append(mag_enhanced.updateMask(mask).rename('mag_enhanced'))
    if dur_cat:
        out.append(dur_cat.updateMask(mask).rename('dur_categorical'))

    return mask, ee.Image.cat(out)

# Reuse existing labels if already built; otherwise build now
if 'labels_by_asset' not in globals() or not labels_by_asset:
    labels_by_asset = {}
    for name, img in images.items():
        try:
            res = _build_categorical_labels(img, magnitude_threshold=170)
            if res:
                labels_by_asset[name] = {'mask': res[0], 'labels': res[1], 'image': img}
                print(f"✓ {name}")
            else:
                print(f"✗ {name}: no MAG band")
        except Exception as e:
            print(f"✗ {name}: {e}")

print(f"\nAssets with labels: {len(labels_by_asset)}")

# Map: toggle per-asset labels
Map_LabelsOnly = geemap.Map(center=[37.5, -98], zoom=4)

vis_yod = {'bands': ['yod_categorical'], 'min': 2016, 'max': 2024,
           'palette': ['navy','blue','cyan','lime','yellow','orange','red','darkred']}
vis_mag = {'bands': ['mag_enhanced'], 'min': -800, 'max': 800,
           'palette': ['darkred','red','orange','white','lightgreen','green','darkgreen']}
vis_dur = {'bands': ['dur_categorical'], 'min': 1, 'max': 8,
           'palette': ['yellow','orange','darkorange','red','darkred','purple','navy']}

for name, rec in labels_by_asset.items():
    bands = rec['labels'].bandNames().getInfo()
    if 'yod_categorical' in bands:
        Map_LabelsOnly.addLayer(rec['labels'].select('yod_categorical'), vis_yod, f"{name} • YOD (mode)", name=='Austin_Urban')
    Map_LabelsOnly.addLayer(rec['labels'].select('mag_enhanced'), vis_mag, f"{name} • MAG (smoothed)", False)
    if 'dur_categorical' in bands:
        Map_LabelsOnly.addLayer(rec['labels'].select('dur_categorical'), vis_dur, f"{name} • DUR (mode)", False)

print("Labels viewer ready. Toggle per-asset YOD/MAG/DUR layers.")
Map_LabelsOnly


Assets with labels: 15
Labels viewer ready. Toggle per-asset YOD/MAG/DUR layers.


Map(center=[37.5, -98], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

## AlphaEarth

In [10]:
# AlphaEarth Analysis - Compare with LandTrendr Labels
print("🌍 Loading AlphaEarth embeddings for Austin area (2017-2024)...")

# Update Austin area to match your AOI bounds: rectFromCenter(-97.8, 30.3, 0.4)
austin_aoi = ee.Geometry.Rectangle([-98.0, 30.1, -97.6, 30.5])  # ~40km width centered on -97.8, 30.3

# Load AlphaEarth embeddings collection
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
years = list(range(2017, 2025))  # 2017-2024

# Load AlphaEarth for each year
alphaearth_images = {}
print("Loading AlphaEarth embeddings...")

for year in years:
    try:
        img = (embeddings
               .filterDate(ee.Date.fromYMD(year, 1, 1), ee.Date.fromYMD(year + 1, 1, 1))
               .filterBounds(austin_aoi)
               .mosaic()
               .clip(austin_aoi))
        alphaearth_images[year] = img
        print(f"✓ Loaded AlphaEarth {year}")
    except Exception as e:
        print(f"✗ Failed to load {year}: {e}")

# Get AlphaEarth band names (should be A01-A64)
if alphaearth_images:
    sample_year = list(alphaearth_images.keys())[0]
    ae_bands = alphaearth_images[sample_year].bandNames().getInfo()
    print(f"AlphaEarth has {len(ae_bands)} embedding dimensions")
    print(f"Band names: {ae_bands[:5]}...{ae_bands[-5:]}")
else:
    print("No AlphaEarth images loaded successfully")

print(f"Ready to analyze AlphaEarth embeddings for {len(alphaearth_images)} years!")

🌍 Loading AlphaEarth embeddings for Austin area (2017-2024)...
Loading AlphaEarth embeddings...
✓ Loaded AlphaEarth 2017
✓ Loaded AlphaEarth 2018
✓ Loaded AlphaEarth 2019
✓ Loaded AlphaEarth 2020
✓ Loaded AlphaEarth 2021
✓ Loaded AlphaEarth 2022
✓ Loaded AlphaEarth 2023
✓ Loaded AlphaEarth 2024
AlphaEarth has 64 embedding dimensions
Band names: ['A00', 'A01', 'A02', 'A03', 'A04']...['A59', 'A60', 'A61', 'A62', 'A63']
Ready to analyze AlphaEarth embeddings for 8 years!


In [11]:
# 1. YEAR OF CHANGE (YOC) from AlphaEarth temporal evolution
print("\n📅 1. GENERATING YEAR OF CHANGE (YOC) LAYER...")

# Calculate temporal differences between consecutive years
temporal_changes = {}
change_magnitudes = {}

for i, year in enumerate(sorted(alphaearth_images.keys())[:-1]):
    next_year = sorted(alphaearth_images.keys())[i + 1]
    
    # Calculate absolute difference in the most temporally sensitive dimension
    temp_diff = alphaearth_images[next_year].select(temporal_sensitive_band).subtract(
        alphaearth_images[year].select(temporal_sensitive_band)).abs()
    
    temporal_changes[year] = temp_diff
    change_magnitudes[year] = temp_diff
    print(f"  ✓ Calculated change {year}→{next_year}")

# Find the year with maximum change for each pixel using iterative approach
print("Finding year of maximum change for each pixel...")

max_change = None
yoc_year = None

for i, year in enumerate(sorted(temporal_changes.keys())):
    if max_change is None:
        max_change = temporal_changes[year]
        yoc_year = ee.Image.constant(year)
    else:
        # Update where current change is greater
        is_greater = temporal_changes[year].gt(max_change)
        max_change = max_change.where(is_greater, temporal_changes[year])
        yoc_year = yoc_year.where(is_greater, year)

yoc_alphaearth = yoc_year.rename('yoc_alphaearth')

print("  ✓ Year of Change layer generated from AlphaEarth temporal evolution")


📅 1. GENERATING YEAR OF CHANGE (YOC) LAYER...


NameError: name 'temporal_sensitive_band' is not defined